In [1]:
here::i_am("revisions/04_TEomes_interaction/01_AllantoisTrajectory_KDvWT.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(destiny))
suppressPackageStartupMessages(library(tradeSeq))
suppressPackageStartupMessages(library(slingshot))
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(BiocParallel))

suppressPackageStartupMessages(library(circlize))
suppressPackageStartupMessages(library(scales))
suppressPackageStartupMessages(library(ComplexHeatmap))

BPPARAM <- BiocParallel::bpparam()
BPPARAM$workers = 24

# Multi core using future - built in to seurat
plan("multicore", workers = 16)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM
set.seed(42)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘rtracklayer’ was built under R version 4.2.3”

                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | 

In [15]:
args = list()

# Metadata
args$metadata = file.path(io$basedir, 'results/rna_atac/clustering/metadata_celltype_annotated_v2.txt.gz')

# RNA_sce
args$rna_sce = file.path(io$basedir, 'results/rna_atac/trajectory/v3/rna_sce.rds')

args$logFC_thr = 0.25
args$FDR_thr = 0.05

# outdir
args$outdir = file.path(io$basedir, 'results/thesis')
dir.create(args$outdir, showWarnings = F, recursive = T)

In [3]:
# Load meta
meta = fread(args$metadata)[day %in% c('D3.5', 'D4', 'D4.5', 'D5')] %>% # Remove D3 for KD vs WT comparison
    .[celltype_v2 %in% c('Primitive_Streak', 'Early_Mes_EOi','Early_Mes_EOd','Posterior_Mes','HE_Precursor', 'HE', 'Allantois_Precursor')]

In [4]:
# load sce
rna.sce <- load_SingleCellExperiment(args$rna_sce, normalise = F, cells = meta$cell)

In [5]:
# Assign cells to lineage, doing this now to make sure it's consistent
# Notice, assignCells is not fixed in which cells are assigned which lineage, even after setting a seed!
set.seed(42)
assign_lineages = tradeSeq:::.assignCells(assays(rna.sce$slingshot)$weights)
#colnames(assign_lineages) = c('Trajectory1', 'Trajectory2')
pseudotime = slingPseudotime(rna.sce$slingshot, na=T)

Traj_assignment = as.data.table(assign_lineages, keep.rownames=T) %>%
    setnames(c('cell', paste0('Trajectory', 1:(length(.)-1))))

Traj1 = as.data.table(pseudotime[,1], keep.rownames=T) %>% 
    .[V1 %in% Traj_assignment[Trajectory1 == 1, cell]] #%>% 
   # .[V2>5 & V2<10]

Traj2 = as.data.table(pseudotime[,2], keep.rownames=T) %>% 
    .[V1 %in% Traj_assignment[Trajectory2 == 1, cell]] #%>% 
    #.[V2>5 & V2<10]

In [6]:
# Combine & add which trajectory
Traj_full = rbind(Traj1[,Traj:='Trajectory1'], Traj2[,Traj:='Trajectory2']) %>%
    setnames(c('cell', 'pseudotime', 'Traj'))

In [12]:
# Add trajectory info to metadata
meta2 = rbind(meta[cell %in% Traj_assignment[Trajectory1==1, cell]] %>% 
                  .[, trajectory := 'Traj1'] %>%
                  .[, pseudotime := colData(rna.sce[,cell])$slingPseudotime_1],
              meta[cell %in% Traj_assignment[Trajectory2==1, cell]] %>%
                  .[, trajectory := 'Traj2'] %>%
                  .[, pseudotime := colData(rna.sce[,cell])$slingPseudotime_2])

# Keep only Allantois trajectory 
meta2 = meta2[trajectory == 'Traj2']

In [26]:
# facet_labels = c()
meta2$genotype = factor(gsub('KO', 'Eomes-KD', meta2$genotype), levels = c('WT', 'Eomes-KD'))
p1 = ggplot(meta2, aes(pseudotime, fill=celltype_v2)) + 
    # geom_histogram(binwidth=0.1, position='stack') + 
    geom_density(aes(y = after_stat(density * n/10)), position = 'stack', color = 'black', linewidth = 0.4, bw = 0.2) + 
    scale_colour_manual(values = opts$days.colors, name='day') + 
    scale_fill_manual(values = opts$celltype_v2.colors, name='Celltype') +
    scale_x_continuous(expand=c(0,0)) + 
    scale_y_continuous(expand=c(0,0)) + 
    geom_vline(xintercept = c(2, 10), color='red', linetype = 'dashed', linewidth = 1) + 
    xlab('Pseudotime') + ylab('Cell density') + 
    facet_wrap(~genotype, ncol = 2, 
               scales='fixed', 
               labeller = labeller(trajectory = c('Traj1' = 'Trajectory 1', 'Traj2' = 'Trajectory 2'))) + 
    theme_classic() + 
    theme(text = element_text(size=30),
          axis.text = element_text(color='black'),
          axis.text.y = element_blank(),
          axis.ticks.y = element_blank(),
          strip.background = element_blank(),
          strip.text = element_text(size=30),
          legend.position='none')
p1_values = ggplot_build(p1)
p1 = p1 +    
    ggrastr::rasterise(geom_segment(aes(x = pseudotime, xend = pseudotime, y = 0, yend = -max(p1_values$data[[1]]$ymax)/10, col=day)), dpi = 2000) + 
    geom_hline(yintercept=c(0,-max(p1_values$data[[1]]$ymax)/10), color='black')

options(repr.plot.width = 5, repr.plot.height=6)
# p1

In [27]:
ggsave(file.path(args$outdir, 'EoT_pseudotime.pdf'), 
       plot = p1,
       width = 190, 
       height = 100, 
       units = "mm")